# Debug Inference Notebook

Minimal, fully manual control over hardware inference.

**Cell order:**
1. Setup — load overlay, connect peripherals, define all constants
2. Create weight matrix and include (n_inc) matrix
3. Write weights / includes to a specific specialist section in BRAM
4. Read back a specialist section from BRAM
5. Run inference for a single specialist — GPIO-polled handshake, print class sums

---

**GPIO map:**

| Channel | Offset | Direction | Bits |
|---------|--------|-----------|------|
| CH1 | `0x000` | PL → PS | bit 0 = `request` |
| CH2 | `0x008` | PS → PL | bits [4:1] = one-hot select, bit 0 = valid |

**BRAM layout (per specialist):**

| Region | Words | Byte offset from specialist base |
|--------|-------|----------------------------------|
| Weights | 80 | `0` – `319` |
| Includes clause 0 | 68 | `320` – `591` |
| Includes clause 1 | 68 | `592` – `863` |
| … | … | … |
| Includes clause 7 | 68 | `2496` – `2767` (last word at `2768`) |
| **Specialist base** | | `spec_idx × 624 × 4` bytes |

In [21]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
import numpy as np
import time
import threading
import random
from pynq import Overlay, allocate, Clocks, MMIO

# ── Overlay ───────────────────────────────────────────────────────────────────
ol      = Overlay('bitstreams/tmc5.bit')
dma     = ol.axi_dma_0
gpio    = ol.axi_gpio_0
bram    = ol.axi_bram_ctrl_0.mmio

# ── GPIO offsets ──────────────────────────────────────────────────────────────
GPIO_CH1 = 0x000   # PL → PS : bit 0 = request (input to PS)
GPIO_CH2 = 0x008   # PS → PL : bits [4:1] one-hot select, bit 0 valid (output from PS)

# ── Architecture constants  (must match VHDL generics in Types.vhd) ───────────
IMG_SIZE        = 32
NUM_PIXELS      = IMG_SIZE * IMG_SIZE       # 1024
NUM_SPECIALISTS = 4
NUM_CLAUSES     = 8
NUM_CLASSES     = 10
MAX_WEIGHT      = 7
ENC_BITS        = 7
POS_BITS        = 29                        # IMG_SIZE - PS0 = 32 - 3
MAX_PS          = 7
WGHT_BITS       = 4                         # clog2(MAX_WEIGHT) + 2 = clog2(7) + 2
SUM_BITS        = 8                         # clog2(MAX_WEIGHT * NUM_CLAUSES) + 1 = clog2(56) + 1
CS_NUM_WORDS    = 3                         # words on m_axis per inference (word0 + word1)

# ── BRAM layout constants ─────────────────────────────────────────────────────
LIT_BITS     = 2 * (2*POS_BITS + 3*MAX_PS*MAX_PS*ENC_BITS)   # 2174
WGHT_WORDS   = NUM_CLAUSES * NUM_CLASSES                      # 80
INC_WORDS    = (LIT_BITS + 31) // 32                          # 68
SPCLST_WORDS = WGHT_WORDS + NUM_CLAUSES * INC_WORDS           # 624

# ── One-hot specialist select ─────────────────────────────────────────────────
SPCLST_SELECT = {0: 0b0001, 1: 0b0010, 2: 0b0100, 3: 0b1000}

# ── Sanity checks ─────────────────────────────────────────────────────────────
assert LIT_BITS == 2174,     f'LIT_BITS mismatch: {LIT_BITS}'
assert INC_WORDS == 68,      f'INC_WORDS mismatch: {INC_WORDS}'
assert SPCLST_WORDS == 624,  f'SPCLST_WORDS mismatch: {SPCLST_WORDS}'
# Note: bram.length reflects the AXI window (8 K) which is less than the
# 4 × 624 × 4 = 9984 bytes needed for all specialists.  The underlying BRAM
# block is larger; PYNQ's MMIO maps enough physical pages for the writes to
# reach all addresses.  Remove this check if it raises an AssertionError.
assert bram.length >= NUM_SPECIALISTS * SPCLST_WORDS * 4, \
    f'BRAM too small: {bram.length} bytes < {NUM_SPECIALISTS * SPCLST_WORDS * 4} needed'

# ── DMA output buffer for class sums (S2MM) ───────────────────────────────────
# 2 × uint32 = 8 bytes; the DMA stops automatically on m_axis TLAST.
# HP port is non-cache-coherent: flush in_buf before MM2S,
# invalidate out_buf after S2MM.
out_buf = allocate(shape=(CS_NUM_WORDS,), dtype=np.uint32)
print(f'out_buf : {out_buf.nbytes} bytes @ 0x{out_buf.physical_address:08x}')

print(f'PL clock     : {Clocks.fclk0_mhz:.1f} MHz')
print(f'BRAM size    : {bram.length} bytes  ({bram.length // 4} words)')
print(f'LIT_BITS     : {LIT_BITS}')
print(f'WGHT_WORDS   : {WGHT_WORDS}  per specialist')
print(f'INC_WORDS    : {INC_WORDS}   per clause')
print(f'SPCLST_WORDS : {SPCLST_WORDS} per specialist')
print('✓ Setup complete')

out_buf : 12 bytes @ 0x1584e000
PL clock     : 10.0 MHz
BRAM size    : 16384 bytes  (4096 words)
LIT_BITS     : 2174
WGHT_WORDS   : 80  per specialist
INC_WORDS    : 68   per clause
SPCLST_WORDS : 624 per specialist
✓ Setup complete


In [22]:
print(gpio.read(GPIO_CH1))

1


In [32]:
# ── Cell 2  (Test Case 1) ──────────────────────────────────────────────────────
#
# All literals excluded → every clause fires on every patch unconditionally.
# Class sum = Σ weights[c, j]  for c = 0..7.
#
# Per-class design:
#   class 0  : all weights  7            → sum =   56  (MAX possible sum)
#   class 1  : all weights -8            → sum =  -64  (MIN possible sum)
#   class 2  : alternating  7 / -8      → sum =   -4  (all-extreme alternation)
#   class 3  : pairs ±7, ±6, ±5, ±4    → sum =    0  (exact cancellation)
#   class 4  : weight  8 in clause 0    → sum =   -8  (overflow: 8 wraps to -8)
#   class 5  : weight -9 in clause 0    → sum =    7  (overflow: -9 wraps to +7)
#   class 6  : descending 7 → 0        → sum =   28  (positive, mid-range)
#   class 7  : descending -1 → -8      → sum =  -36  (full negative range)
#   class 8  : mixed signs              → sum =    2  (general accumulation)
#   class 9  : mixed signs              → sum =    3  (general accumulation)
#
# Overflow mechanics (write_specialist masks to 4 bits):
#    8  & 0xF = 0b1000  → VHDL signed = -8
#   -9  & 0xF = 0b0111  → VHDL signed = +7
# ──────────────────────────────────────────────────────────────────────────────

weights = np.zeros((NUM_CLAUSES, NUM_CLASSES), dtype=np.int32)
n_inc   = np.zeros((NUM_CLAUSES, LIT_BITS),   dtype=bool)

# ── Weights ────────────────────────────────────────────────────────────────────
#                         c0   c1   c2   c3   c4   c5   c6   c7   c8   c9
weights[0, :] = np.array([ 7,  -8,   7,   7,   0,   7,  -1,   7,   random.randint(-8, 7), random.randint(-8, 7)])
weights[1, :] = np.array([ 7,  -8,  -8,  -7,   0,   6,  -2,  -8,   random.randint(-8, 7), random.randint(-8, 7)])
weights[2, :] = np.array([ 7,  -8,   7,   6,   0,   5,  -3,   5,   random.randint(-8, 7), random.randint(-8, 7)])
weights[3, :] = np.array([ 7,  -8,  -8,  -6,   0,   4,  -4,  -6,   random.randint(-8, 7), random.randint(-8, 7)])
weights[4, :] = np.array([ 7,  -8,   7,   5,   0,   3,  -5,   3,   random.randint(-8, 7), random.randint(-8, 7)])
weights[5, :] = np.array([ 7,  -8,  -8,  -5,   0,   2,  -6,  -4,   random.randint(-8, 7), random.randint(-8, 7)])
weights[6, :] = np.array([ 7,  -8,   7,   4,   0,   1,  -7,   1,   random.randint(-8, 7), random.randint(-8, 7)])
weights[7, :] = np.array([ 7,  -8,  -8,  -4,   0,   0,  -8,  -2,   random.randint(-8, 7), random.randint(-8, 7)])

# ── Include mask ───────────────────────────────────────────────────────────────
n_inc[:, :] = True   # all literals excluded → all clauses fire unconditionally

# ── Python oracle ──────────────────────────────────────────────────────────────
# write_specialist masks each weight to 4 bits before writing to BRAM.
# The oracle must apply the same masking so overflow cases are predicted correctly.
def hw_weight(w):
    raw = int(w) & 0xF
    return raw - 16 if raw >= 8 else raw

hw_weights    = np.vectorize(hw_weight)(weights)
expected_sums = hw_weights.sum(axis=0).tolist()
# → [56, -64, -4, 0, -8, 7, 28, -36, 2, 3]

# ── Summary ────────────────────────────────────────────────────────────────────
print('hw_weights (after 4-bit wrap):')
print(hw_weights)
print(f'\nn_inc: all excluded (always fires) : {n_inc.all()}')
print(f'\nExpected class sums : {expected_sums}')
print(f'  class 0 =  56  (MAX sum: 8 clauses × +7)')
print(f'  class 1 = -64  (MIN sum: 8 clauses × -8)')
print(f'  class 2 =  -4  (alternating extremes)')
print(f'  class 3 =   0  (exact sign cancellation)')
print(f'  class 4 =  -8  (overflow: weight 8 stored as -8)')
print(f'  class 5 =   7  (overflow: weight -9 stored as +7)')
print(f'  class 6 =  28  (descending 7→0, positive)')
print(f'  class 7 = -36  (descending -1→-8, sign extension)')

hw_weights (after 4-bit wrap):
[[ 7 -8  7  7  0  7 -1  7  0 -2]
 [ 7 -8 -8 -7  0  6 -2 -8  5  2]
 [ 7 -8  7  6  0  5 -3  5  2 -5]
 [ 7 -8 -8 -6  0  4 -4 -6  2  2]
 [ 7 -8  7  5  0  3 -5  3  4  0]
 [ 7 -8 -8 -5  0  2 -6 -4 -8 -6]
 [ 7 -8  7  4  0  1 -7  1 -5 -3]
 [ 7 -8 -8 -4  0  0 -8 -2  0 -2]]

n_inc: all excluded (always fires) : True

Expected class sums : [56, -64, -4, 0, 0, 28, -36, -4, 0, -14]
  class 0 =  56  (MAX sum: 8 clauses × +7)
  class 1 = -64  (MIN sum: 8 clauses × -8)
  class 2 =  -4  (alternating extremes)
  class 3 =   0  (exact sign cancellation)
  class 4 =  -8  (overflow: weight 8 stored as -8)
  class 5 =   7  (overflow: weight -9 stored as +7)
  class 6 =  28  (descending 7→0, positive)
  class 7 = -36  (descending -1→-8, sign extension)


In [33]:
# ── Cell 3: Write to BRAM ─────────────────────────────────────────────────────
#
# Pass weights=None to skip writing weights.
# Pass n_inc=None to skip writing includes.
# ──────────────────────────────────────────────────────────────────────────────

def _pack_n_inc(n_inc_clause):
    """Pack one clause's n_inc bool vector into INC_WORDS uint32 words.
    Bit 0 of word 0 = literal 0.  (little-endian, matches VHDL.)"""
    padded = np.zeros(INC_WORDS * 32, dtype=np.uint8)
    padded[:LIT_BITS] = n_inc_clause.astype(np.uint8)
    return np.packbits(padded, bitorder='little').view(np.uint32)   # INC_WORDS words


def write_specialist(spec_idx, weights=None, n_inc=None):
    """Write weights and/or n_inc for specialist spec_idx to BRAM."""
    base_byte = spec_idx * SPCLST_WORDS * 4

    if weights is not None:
        for c in range(NUM_CLAUSES):
            for j in range(NUM_CLASSES):
                offset = base_byte + (c * NUM_CLASSES + j) * 4
                bram.write(offset, int(weights[c, j]) & 0xFFFFFFFF)
        print(f'Spec {spec_idx}: weights written  '
              f'(base byte {base_byte}, words 0–{WGHT_WORDS - 1})')

    if n_inc is not None:
        for c in range(NUM_CLAUSES):
            words    = _pack_n_inc(n_inc[c])
            cl_base  = base_byte + (WGHT_WORDS + c * INC_WORDS) * 4
            for k, w in enumerate(words):
                bram.write(cl_base + k * 4, int(w))
        print(f'Spec {spec_idx}: includes written '
              f'(words {WGHT_WORDS}–{SPCLST_WORDS - 1})')

    if weights is None and n_inc is None:
        print('Nothing written — both weights and n_inc are None.')


# ── Edit here ─────────────────────────────────────────────────────────────────
write_specialist(2, weights=weights, n_inc=n_inc)


# write_specialist(SPEC_WRITE, weights=weights)   # weights only
# write_specialist(SPEC_WRITE, n_inc=n_inc)       # includes only

Spec 2: weights written  (base byte 4992, words 0–79)
Spec 2: includes written (words 80–623)


In [34]:
# ── Cell 4: Read back one specialist from BRAM ────────────────────────────────

def read_specialist(spec_idx):
    base_byte = spec_idx * SPCLST_WORDS * 4
    base_word = spec_idx * SPCLST_WORDS

    print(f'=== Specialist {spec_idx}  (base word {base_word}, base byte {base_byte}) ===')

    # ── Weights ───────────────────────────────────────────────────────────────
    # VHDL reads lower WGHT_BITS (4) bits as signed.
    # Mask with 0xF, then sign-extend from 4 bits.
    print('\nWeights  (rows = clauses, cols = classes):')
    w_rd = np.zeros((NUM_CLAUSES, NUM_CLASSES), dtype=np.int32)
    for c in range(NUM_CLAUSES):
        for j in range(NUM_CLASSES):
            offset = base_byte + (c * NUM_CLASSES + j) * 4
            raw    = bram.read(offset) & 0xF       # lower 4 bits
            w_rd[c, j] = raw - 16 if raw >= 8 else raw
    print(w_rd)

    # ── Includes ──────────────────────────────────────────────────────────────
    # Show first word and last word of each clause's include region.
    print(f'\nIncludes  (word[0] and word[{INC_WORDS - 1}] per clause):')
    print(f'  {"clause":<8}  {"word[0]":>12}  {"word[last]":>12}  {"all excluded?":>14}')
    for c in range(NUM_CLAUSES):
        cl_base  = base_byte + (WGHT_WORDS + c * INC_WORDS) * 4
        words    = np.array([bram.read(cl_base + k * 4) for k in range(INC_WORDS)],
                            dtype=np.uint32)
        # Unpack and check
        bits     = np.unpackbits(words.view(np.uint8), bitorder='little')[:LIT_BITS]
        all_excl = bool(bits.all())
        print(f'  {c:<8}  0x{words[0]:08x}    0x{words[-1]:08x}    {str(all_excl):>14}')


# ── Edit here ─────────────────────────────────────────────────────────────────
SPEC_READ = 0

read_specialist(0)
read_specialist(1)
read_specialist(2)
read_specialist(3)


=== Specialist 0  (base word 0, base byte 0) ===

Weights  (rows = clauses, cols = classes):
[[ 7 -8  7  7  0  7 -1  7 -5  5]
 [ 7 -8 -8 -7  0  6 -2 -8 -2  4]
 [ 7 -8  7  6  0  5 -3  5 -1 -4]
 [ 7 -8 -8 -6  0  4 -4 -6  5 -2]
 [ 7 -8  7  5  0  3 -5  3 -7 -5]
 [ 7 -8 -8 -5  0  2 -6 -4  2  7]
 [ 7 -8  7  4  0  1 -7  1  0 -1]
 [ 7 -8 -8 -4  0  0 -8 -2 -1 -8]]

Includes  (word[0] and word[67] per clause):
  clause         word[0]    word[last]   all excluded?
  0         0xffffffff    0x3fffffff              True
  1         0xffffffff    0x3fffffff              True
  2         0xffffffff    0x3fffffff              True
  3         0xffffffff    0x3fffffff              True
  4         0xffffffff    0x3fffffff              True
  5         0xffffffff    0x3fffffff              True
  6         0xffffffff    0x3fffffff              True
  7         0xffffffff    0x3fffffff              True
=== Specialist 1  (base word 624, base byte 2496) ===

Weights  (rows = clauses, cols = classes):
[[ 

In [35]:
# ── Cell 5: Run inference for ONE specialist ───────────────────────────────────
#
# GPIO handshake:
#   1. Assert one-hot select + valid on CH2.
#   2. Poll CH1 until request goes LOW  (hardware has latched select → S_LOAD).
#   3. Immediately deassert CH2.
#   4. Fixed delay while S_LOAD completes (~624 cycles @ 50 MHz = 12.5 µs).
#   5. Arm DMA S2MM (receive) so it is ready when m_axis fires after patches.
#   6. Stream all pixels via DMA MM2S; poll sendchannel.idle (no MM2S interrupt).
#   7. Wait for S2MM interrupt (s2mm_introut → IRQ_F2P[1]) — class sums in out_buf.
#   8. Invalidate out_buf cache (HP port, non-coherent) and decode class sums.
# ──────────────────────────────────────────────────────────────────────────────

# ── GPIO helpers ──────────────────────────────────────────────────────────────
def gpio_assert(spec_idx):
    """Assert one-hot select + valid on CH2."""
    sel = SPCLST_SELECT[spec_idx]
    gpio.write(GPIO_CH2, (sel << 1) | 1)

def gpio_deassert():
    """Deassert CH2 (valid=0, select=0)."""
    gpio.write(GPIO_CH2, 0)

def gpio_request():
    """Return current value of request bit from PL (CH1 bit 0)."""
    return (gpio.read(GPIO_CH1) & 1)

def decode_sums(reg0, reg1, reg2):
    raw  = (int(reg2) << 64) | (int(reg1) << 32) | int(reg0)
    mask = (1 << SUM_BITS) - 1
    sums = []
    for i in range(NUM_CLASSES):
        v = (raw >> (i * SUM_BITS)) & mask
        if v >= (1 << (SUM_BITS - 1)):
            v -= (1 << SUM_BITS)
        sums.append(v)
    return sums


# ── Edit here ─────────────────────────────────────────────────────────────────
SPEC_RUN    = 2        # specialist index to run (0–3)
PIX_FILE    = 'data/pixel_input.txt'   # pixel source
# To use a flat / uniform pixel pattern instead, comment the file block below
# and set:  pixel_words = np.full(NUM_PIXELS, 0x00808080, dtype=np.uint32)

# ── Build pixel buffer ────────────────────────────────────────────────────────
with open(PIX_FILE) as f:
    lines = [l.strip() for l in f if l.strip() and not l.startswith('#')]
assert len(lines) >= NUM_PIXELS, f'Need {NUM_PIXELS} pixels, got {len(lines)}'


pixel_words_white = np.full(49, 0x00FFFFFF, dtype=np.uint32)   # all-white → all ENC bits = 1
pixel_words_black = np.zeros(NUM_PIXELS - 49, dtype=np.uint32) # all-black → all ENC bits = 0

in_buf      = allocate(shape=(NUM_PIXELS,), dtype=np.uint32)
in_buf[:]   = np.concatenate([pixel_words_black[:NUM_PIXELS//2], pixel_words_white, pixel_words_black[NUM_PIXELS//2:]])
in_buf.flush()   # HP port: write CPU cache → DDR before DMA reads
print(f'Pixel buffer : {in_buf.nbytes} bytes @ 0x{in_buf.physical_address:08x}')
print(f'First word   : 0x{pixel_words_white[0]:08x}')
print(f'Last  word   : 0x{pixel_words_black[-1]:08x}')

# ── Run inference ─────────────────────────────────────────────────────────────
print(f'\nRunning inference — Specialist {SPEC_RUN} ...')

# 1. Wait for hardware to assert request before sending selection
print("Waiting for GPIO request...")
t0 = time.time()
while not gpio_request():
    if time.time() - t0 > 5.0:
        raise TimeoutError('Hardware never asserted request — is the bitstream loaded?')
    time.sleep(0.001)
print("GPIO request:", gpio_request())

gpio_assert(SPEC_RUN)
print("Asserted select + valid on GPIO.")

# 2. Poll until request goes low (hardware latched select, now in S_LOAD)
t0 = time.time()
while gpio_request():
    if time.time() - t0 > 2.0:
        gpio_deassert()
        raise TimeoutError('Timed out waiting for request to go low — is the hardware running?')

# 3. Immediately deassert valid
gpio_deassert()

# 4. Wait for S_LOAD to complete
# SPCLST_WORDS=624 cycles @ 50 MHz = 12.5 µs; 1 ms is very conservative.
time.sleep(0.1)

# 5. Arm S2MM (receive) BEFORE starting MM2S
# Class sums arrive on m_axis after all 676 patches are processed, well after
# MM2S finishes.  Arming first guarantees DMA is listening when m_axis fires.
out_buf[:] = 0
out_buf.flush()   # clear any stale DDR content
dma.recvchannel.start()
dma.recvchannel.transfer(out_buf)   # 8 bytes; stops on TLAST from m_axis

# 6. Stream pixels via DMA MM2S
t1 = time.time()
dma.sendchannel.start()
dma.sendchannel.transfer(in_buf)
# Poll MM2S idle — mm2s_introut is not connected in this block design
while not dma.sendchannel.idle:
    if time.time() - t1 > 5.0:
        raise TimeoutError('DMA send timeout')
    time.sleep(0.001)
print(f'  MM2S done in {(time.time()-t1)*1e3:.2f} ms')

# 7. Wait for S2MM interrupt (s2mm_introut → IRQ_F2P[1])
# Use a daemon thread so we can apply a safety timeout.
s2mm_done = threading.Event()
def _wait_s2mm():
    try:
        dma.recvchannel.wait()
    finally:
        s2mm_done.set()
threading.Thread(target=_wait_s2mm, daemon=True).start()
if not s2mm_done.wait(timeout=10.0):
    print('  ⚠ S2MM interrupt timeout — falling back to polling')
    while not dma.recvchannel.idle:
        time.sleep(0.001)
print(f'  S2MM done')

# 8. Invalidate out_buf cache (HP port: re-read from DDR) and decode
out_buf.invalidate()
sums = decode_sums(out_buf[0], out_buf[1], out_buf[2])

print(f'\n  out_buf[0] = 0x{int(out_buf[0]):08x}')
print(f'  out_buf[1] = 0x{int(out_buf[1]):08x}')
print(f'  out_buf[2] = 0x{int(out_buf[2]):08x}')
print(f'\nExpected class sums\t: {expected_sums}')
print(f'Read out class sums\t: {sums}')
print(f'Predicted\t\t: class {int(np.argmax(sums))}')

Pixel buffer : 4096 bytes @ 0x1584c000
First word   : 0x00ffffff
Last  word   : 0x00000000

Running inference — Specialist 2 ...
Waiting for GPIO request...
GPIO request: 1
Asserted select + valid on GPIO.
  MM2S done in 3.27 ms
  S2MM done

  out_buf[0] = 0x00fcc038
  out_buf[1] = 0xfcdc1c00
  out_buf[2] = 0x0000f200

Expected class sums	: [56, -64, -4, 0, 0, 28, -36, -4, 0, -14]
Read out class sums	: [56, -64, -4, 0, 0, 28, -36, -4, 0, -14]
Predicted		: class 0


In [87]:
print(gpio.read(GPIO_CH1))

1
